# Project: European Air Traffic Pipeline on EUROCONTROL OPDI data (Medallion Architecture, PySpark)

## DAT535 - Data Engineering

### Thomas Canisares--Morère & Mathis Nancey - Group 08

Batch pipeline that turns EUROCONTROL OPDI flight data into clean, analysis-ready datasets using the Medallion Architecture.

- Period: June 1, 2026 -> July 28, 2026 (~5.5 GB)
- Tables: flight_list (monthly), flight_events and measurements (10-day intervals), OurAirports repositories (airports, runways)

## Part 1: Setup
### Part 1.1: Imports

In [ ]:
import os
import json
import time
import datetime
import random
import csv
import hashlib
import shutil
import urllib.error
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lit, when, expr, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, year, month, hour, dayofweek,
    lower, upper, trim, length, concat, concat_ws, coalesce, explode, split,
    countDistinct, collect_list, first, row_number, unix_timestamp,
    from_json, map_keys, radians, sin, cos, asin, sqrt,
    input_file_name, regexp_extract, current_timestamp, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType, MapType
)

print("Libraries imported successfully")

### Part 1.2: SparkSession

In [ ]:
# SparkSession
spark = SparkSession.builder \
    .appName("DAT535-Project_Thomas_Mathis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


### Part 1.3: Parameters

In [ ]:
START = datetime.date(2026, 6, 1)
END   = datetime.date(2026, 7, 29)
ORIGIN     = datetime.date(2022, 1, 1)            # the 10-day periods are counted starting from this date
MIN_FREE_GB = 4.0                                 # minimum free space in GB before downloading a file
RETRIES     = 3                                   # number of download attempts before giving up
PAUSE_S     = 0.5                                 # pause between downloads in seconds

UA = {"User-Agent": "UiS-DAT535-group08"}         # user agent for HTTP requests
BASE_URL   = "https://www.eurocontrol.int/performance/data/download/OPDI/v002"
REF_URLS   = {"airports.csv": "https://davidmegginson.github.io/ourairports-data/airports.csv",
              "runways.csv":  "https://davidmegginson.github.io/ourairports-data/runways.csv"}

PARQUET_TABLES = ["flight_list", "flight_events", "measurements"]

DATA_ROOT      = Path.home() / "opdi-data"
RAW_DIR        = DATA_ROOT / "raw"                # source files as downloaded
BRONZE_DIR     = DATA_ROOT / "bronze"             # raw data + lineage columns
SILVER_DIR     = DATA_ROOT / "silver"
QUARANTINE_DIR = DATA_ROOT / "quarantine"
GOLD_DIR       = DATA_ROOT / "gold"

## Part 2: Bronze Layer
### Part 2.1: Retrieve the data
#### Part 2.1.1: utility functions

In [ ]:
# months() generates the first day of each month in [start, end[
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
def months(start, end):
    m = datetime.date(start.year, start.month, 1)
    while m < end:
        yield m
        m = datetime.date(m.year + (m.month == 12), m.month % 12 + 1, 1)

# Generate 10-day windows starting from ORIGIN
#   - start: the start date (inclusive)
#   - end: the end date (exclusive)
#   - days: the number of days in each window
def windows(start, end, days=10):
    w = ORIGIN + datetime.timedelta(days=((start - ORIGIN).days // days) * days)
    while w < end:
        yield w, w + datetime.timedelta(days=days)
        w += datetime.timedelta(days=days)

# We return here the HTTP status, the expected size in bytes, and the last modified date (if available)
#   - url: the URL to check
def head(url):
    req = urllib.request.Request(url, method="HEAD", headers=UA)
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, int(r.headers.get("Content-Length") or 0), r.headers.get("Last-Modified", "")
    except urllib.error.HTTPError as e:
        return e.code, 0, ""
    except Exception as e:
        return type(e).__name__, 0, ""

# this function downloads the file from the given URL to the destination path.
#   - url: the URL to download from
#   - dest: the destination path (Path object)
def download(url, dest):
    tmp = dest.with_name(dest.name + ".part") # temporary file during download
    for attempt in range(1, RETRIES + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20) # copy in 1 MB chunks
            os.replace(tmp, dest)
            return "downloaded"
        except Exception as e:
            print(f"    attempt {attempt}/{RETRIES} failed : {e}")
            tmp.unlink(missing_ok=True)
            time.sleep(5 * attempt)
    return "failed"

# Compute the SHA256 hash of a file at the given path
#   - path: the path to the file (Path object)
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Return the free disk space in GB for the RAW_DIR
def free_gb():
    return shutil.disk_usage(RAW_DIR).free / 1e9

# Return the current UTC time as a datetime object 
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)

#### Part 2.1.2: List of files

In [ ]:
files = []
for m in months(START, END):
    n = f"flight_list_{m:%Y%m}.parquet"
    files.append(("flight_list", n, f"{BASE_URL}/flight_list/{n}"))
for table in ("flight_events", "measurements"):
    for w, w_end in windows(START, END):
        n = f"{table}_{w:%Y%m%d}_{w_end:%Y%m%d}.parquet"
        files.append((table, n, f"{BASE_URL}/{table}/{n}"))
for n, url in REF_URLS.items():
    files.append(("reference", n, url))

#### Part 2.1.3: Download

The main loop that downloads the files and records the results in a manifest CSV file.
The manifest contains all the relevant information about each file downloaded and it contains the following columns:
- run_id: a unique identifier for this run of the script
- table: the table name (flight_list, flight_events, measurements, reference)
- file: the file name
- url: the URL from which the file was downloaded
- http_status: the HTTP status code returned by the server
- size_bytes: the size of the file in bytes
- last_modified: the last modified date of the file on the server (if available)
- result: the result of the download (downloaded, already_present, absent, empty, failed)
- sha256: the SHA256 hash of the file
- local_path: the local path of the downloaded file
- retrieved_at_utc: the UTC time when the file was retrieved

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RAW_DIR / "manifest.csv"
run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")
print(f"Period [{START}, {END}[ | {len(files)} files | destination {RAW_DIR} | free {free_gb():.1f} GB\n")
rows, t0 = [], time.time() # list to store the results and the start time
for i, (table, name, url) in enumerate(files, 1):
    status, size, last_modified = head(url)
    dest = RAW_DIR / table / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if status != 200:
        result = "absent"
    elif size == 0:
        result = "empty"
    elif dest.exists():
        result = "already_present"
    elif free_gb() - size / 1e9 < MIN_FREE_GB:
        print(f"STOP : insufficient free space before {name}")
        break
    else:
        result = download(url, dest)
    ok = result in ("downloaded", "already_present")
    rows.append(dict(run_id=run_id, table=table, file=name, url=url, http_status=status,
                     size_bytes=size, last_modified=last_modified, result=result,
                     sha256=sha256(dest) if ok else "", local_path=str(dest) if ok else "",
                     retrieved_at_utc=utc_now().isoformat(timespec="seconds")))
    print(f"[{i:2d}/{len(files)}] {table:13s} {name:45s} {size/1e6:8.1f} MB  {result}")
    time.sleep(PAUSE_S)

if rows: # write the manifest CSV file
    new = not manifest_path.exists()
    with open(manifest_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        if new:
            w.writeheader()
        w.writerows(rows)

elapsed = time.time() - t0 

#### Part 2.1.4: Summary

In [ ]:
print(f"\nResults : {dict(Counter(r['result'] for r in rows))}") # print summary of results
for t in ("flight_list", "flight_events", "measurements", "reference"): # print summary of files downloaded for each table
    sel = [r for r in rows if r["table"] == t and r["local_path"]]
    print(f"  {t:13s} {len(sel):3d} files  {sum(int(r['size_bytes']) for r in sel)/1e9:6.2f} GB")
print(f"Duration {elapsed:.0f} s | free space left {free_gb():.1f} GB | manifest: {manifest_path}") # duration, free space, manifest path
for r in rows: # print any files that were not downloaded successfully
    if r["result"] not in ("downloaded", "already_present"):
        print(f"  ! {r['file']} -> {r['result']} (HTTP {r['http_status']})")

### Part 2.2: Schema Validation

OPDI files from different periods were observed with different schemas (e.g. `long` vs `decimal(20,0)`
identifiers, dates stored as `string` vs `timestamp`). Before loading anything into Silver, we check that
all the files of a same table share the same schema, and we record each file's schema fingerprint and
row count in `schema_report.csv` for traceability.

In [ ]:
schema_variants = defaultdict(lambda: defaultdict(list))    # group files according to their schema
report_rows = []                                            # one row per file, written to schema_report.csv

for table in PARQUET_TABLES:
    for path in sorted((RAW_DIR / table).glob("*.parquet")):
        df = spark.read.parquet(str(path))                  # reads only the metadata of the Parquet file
        schema = tuple((f.name, f.dataType.simpleString()) for f in df.schema.fields)
        n_rows = df.count()                                
        fingerprint = hashlib.sha256(str(schema).encode()).hexdigest()[:12]
        schema_variants[table][schema].append(path.name)
        report_rows.append(dict(table=table, file=path.name, n_rows=n_rows, n_columns=len(schema),
                                schema_fingerprint=fingerprint,
                                checked_at_utc=utc_now().isoformat(timespec="seconds")))

for table, variants in schema_variants.items():
    keys = list(variants)
    n_files = sum(len(v) for v in variants.values())
    print(f"\n=== {table}: {n_files} files | {len(keys)} schema variant(s) ===")
    for k, key in enumerate(keys, 1):
        names = variants[key]
        listed = ", ".join(names) if len(names) <= 3 else f"{names[0]} ... {names[-1]}"
        print(f"  Variant {k}: {len(names)} file(s) -> {listed}")
    if len(keys) == 1:                                       # single schema: show it
        for name, dtype in keys[0]:
            print(f"      {name:22s} {dtype}")
    else:                                                    # several schemas: show only the differences
        types = [dict(key) for key in keys]
        all_cols = list(dict.fromkeys(c for t in types for c in t))
        print("  Columns that differ between variants:")
        for c in all_cols:
            vals = [t.get(c, "-- missing --") for t in types]
            if len(set(vals)) > 1:
                print(f"      {c:22s} " + " | ".join(f"V{i + 1}: {v}" for i, v in enumerate(vals)))

# Row counts per file
print("\n=== Rows per file ===")
for r in report_rows:
    print(f"  {r['table']:13s} {r['file']:45s} {r['n_rows']:>12,} rows  schema {r['schema_fingerprint']}")

# Save the report
schema_report_path = RAW_DIR / "schema_report.csv"
with open(schema_report_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(report_rows[0]))
    w.writeheader()
    w.writerows(report_rows)
print(f"\nSchema report written to: {schema_report_path}")

### Part 2.3: Bronze tables (row-level lineage)

The source files in `raw/` are kept untouched. The Bronze tables in `bronze/<table>/` contain
exactly the same rows, columns and types, plus row-level lineage columns taken from the manifest:

| Column | Meaning |
|---|---|
| `_source_file` | name of the raw file the row comes from |
| `_source_period` | month (`YYYYMM`) or 10-day window start (`YYYYMMDD`) of the source file — also the partition key |
| `_ingestion_run_id` / `_ingested_at_utc` | download run that retrieved the file, and when |
| `_source_last_modified` | publication date of the file on the EUROCONTROL server |
| `_source_sha256` | fingerprint of the source file |
| `_bronze_run_id` / `_bronze_loaded_at_utc` | run that built this Bronze table, and when |

No cleaning or type conversion happens here. Tables are partitioned by source period, so re-running only
replaces the partitions of the current period. A reconciliation then checks that every raw row is in Bronze.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")   # a re-run only replaces the partitions it writes

# 1) File-level lineage: latest successful manifest entry for each file
latest = {}
with open(manifest_path, newline="") as f:
    for r in csv.DictReader(f):
        if r["result"] in ("downloaded", "already_present"):
            latest[r["file"]] = r                                    # chronological file: the last entry wins

lineage_df = spark.createDataFrame(
    [(name, r["run_id"], r["retrieved_at_utc"], r["last_modified"], r["sha256"]) for name, r in latest.items()],
    "_source_file string, _ingestion_run_id string, _ingested_at_utc string, "
    "_source_last_modified string, _source_sha256 string",
).withColumn("_ingested_at_utc", col("_ingested_at_utc").cast("timestamp"))

bronze_run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")

# 2) Row-level lineage: each row gets the name of its source file, then the file metadata from the manifest
def add_lineage(df):
    return (df
        .withColumn("_source_file", regexp_extract(input_file_name(), r"([^/]+)$", 1))
        .withColumn("_source_period", regexp_extract(col("_source_file"), r"_(\d{6,8})(?:_\d{8})?\.parquet$", 1))
        .join(broadcast(lineage_df), "_source_file", "left")            # small table: broadcast join, no shuffle
        .withColumn("_bronze_run_id", lit(bronze_run_id))
        .withColumn("_bronze_loaded_at_utc", current_timestamp()))

# Files of the current period only (list built in 2.1.2)
in_scope = defaultdict(list)
for table, name, _ in files:
    if (RAW_DIR / table / name).exists():
        in_scope[table].append(str(RAW_DIR / table / name))

# 3) OPDI tables: original columns and types unchanged, partitioned by source period
for table in PARQUET_TABLES:
    t_start = time.time()
    (add_lineage(spark.read.parquet(*in_scope[table]))
        .write.mode("overwrite")
        .partitionBy("_source_period")
        .parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}  ({time.time() - t_start:.0f} s)")

# 4) Reference tables (OurAirports CSV): every column kept as a string, exactly as in the source file
for path in in_scope["reference"]:
    table = Path(path).stem                                            # airports, runways
    raw_ref = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    (add_lineage(raw_ref).drop("_source_period")
        .write.mode("overwrite").parquet(str(BRONZE_DIR / table)))
    print(f"{table:13s} -> {BRONZE_DIR / table}")

In [ ]:
expected = {}                                                          # raw row counts per file
with open(RAW_DIR / "schema_report.csv", newline="") as f:               # written in 2.2 (Parquet files)
    for r in csv.DictReader(f):
        expected[r["file"]] = int(r["n_rows"])
for path in in_scope["reference"]:                                     # CSV files: counted directly
    expected[Path(path).name] = (spark.read.option("header", True).option("multiLine", True)
                                 .option("escape", '"').csv(path).count())

recon = []
for table in PARQUET_TABLES + [Path(p).stem for p in in_scope["reference"]]:
    stats = (spark.read.parquet(str(BRONZE_DIR / table))
             .groupBy("_source_file")
             .agg(count("*").alias("n_rows"),
                  spark_sum(col("_source_sha256").isNull().cast("int")).alias("missing_lineage"))
             .collect())
    for s in stats:
        recon.append(dict(table=table, file=s["_source_file"], raw_rows=expected.get(s["_source_file"]),
                          bronze_rows=s["n_rows"], missing_lineage=s["missing_lineage"]))

print(f"{'table':13s} {'file':45s} {'raw rows':>12s} {'bronze rows':>12s}  status")
for r in sorted(recon, key=lambda r: (r["table"], r["file"])):
    ok = r["raw_rows"] == r["bronze_rows"] and r["missing_lineage"] == 0
    r["status"] = "OK" if ok else "MISMATCH"
    print(f"{r['table']:13s} {r['file']:45s} {r['raw_rows'] or 0:>12,} {r['bronze_rows']:>12,}  {r['status']}")

n_files_expected = sum(len(v) for v in in_scope.values())
all_ok = len(recon) == n_files_expected and all(r["status"] == "OK" for r in recon)
bronze_gb = sum(p.stat().st_size for p in BRONZE_DIR.rglob("*.parquet")) / 1e9
print(f"\nFiles reconciled: {len(recon)}/{n_files_expected} | rows in Bronze: {sum(r['bronze_rows'] for r in recon):,}"
      f" | Bronze size: {bronze_gb:.2f} GB | free space left: {free_gb():.1f} GB")
print("Reconciliation:", "PASSED" if all_ok else "FAILED")

### Part 2.4: Bronze Summary

**Scope:** June 1, 2026 -> July 28, 2026 (EUROCONTROL OPDI v0.0.2 + OurAirports reference data).

| Table | Source files | Partitions (`_source_period`) | Rows | Raw size |
|---|---|---|---|---|
| flight_list | 2 monthly files | 2 | 3,304,717 | 0.10 GB |
| flight_events | 6 ten-day windows | 6 | 86,670,133 | 3.03 GB |
| measurements | 6 ten-day windows | 6 | 118,779,222 | 2.39 GB |
| airports | 1 CSV | – | 86,258 | 0.02 GB (both CSVs) |
| runways | 1 CSV | – | 48,345 | |
| **Total** | **16 files** | | **208,888,675** | **~5.5 GB** |

**Key Bronze layer properties**
1. **Schema preservation:** original columns and types are kept unchanged (no cleaning, no casting).
2. **Row-level lineage:** every record carries `_source_file`, `_source_period`, `_ingestion_run_id`, `_ingested_at_utc`,
   `_source_last_modified`, `_source_sha256`, `_bronze_run_id` and `_bronze_loaded_at_utc`.
3. **Reconciliation: PASSED.** 16/16 files have the same row count in `raw/` and `bronze/`, and no row is missing its lineage.
4. **Partitioning:** by `_source_period`, with dynamic partition overwrite, so a re-run only replaces the periods it writes.
5. **Storage:** Bronze takes 5.46 GB, slightly *less* than the raw files. Lineage values are constant within a file,
   so Parquet dictionary encoding stores them almost for free.

**Findings that drive the Silver layer**
- One schema variant per table for the whole period: no harmonisation needed (older OPDI periods, e.g. 2025,
  use `decimal` ids, `timestamp_ntz` dates and an extra `unix_time` column and would need it).
- Dates and times (`first_seen`, `last_seen`, `event_time`) are **strings** and must be cast in Silver.
- **Period boundaries:** the first event window starts on May 30 (before the scope) and the last one ends on July 28,
  while `flight_list_202607` covers all of July. Silver must filter on the scope dates.
- `info` (flight_events) is a JSON string; `adep`/`ades` are often missing (NULL or `""`); callsigns can be padded;
  some altitudes are negative.

**Known limitation:** a file already present in `raw/` is not downloaded again. If EUROCONTROL republishes a file
(this happened for 4 event files in October 2026), the change is not detected. Comparing the server `Last-Modified`
with the manifest would fix this.

## Part 3: Silver Layer
### Part 3.1: Data understanding

In the Silver layer, our objective is to transform raw Bronze tables into clean, validated, standardized, deduplicated, and enriched datasets ready for analytics and Gold aggregations.

Before implementing transformations, we profile the Bronze tables to evaluate:
- Volume and schema compatibility across tables
- Null rates on key flight attributes (`adep`, `ades`, `first_seen`, `last_seen`, `flt_id`, `typecode`)
- Temporal anomalies (such as inverted timestamps `first_seen > last_seen`)
- Duplicate keys and referential integrity between `flight_events` and `flight_list`


In [ ]:
bronze_flights_df  = spark.read.parquet(str(BRONZE_DIR / "flight_list"))
bronze_events_df   = spark.read.parquet(str(BRONZE_DIR / "flight_events"))
bronze_airports_df = spark.read.parquet(str(BRONZE_DIR / "airports"))
bronze_runways_df  = spark.read.parquet(str(BRONZE_DIR / "runways"))

flights_count = bronze_flights_df.count()
events_count  = bronze_events_df.count()
print(f"Bronze flight_list   : {flights_count:>12,} rows")
print(f"Bronze flight_events : {events_count:>12,} rows")
print(f"Bronze airports      : {bronze_airports_df.count():>12,} rows")
print(f"Bronze runways       : {bronze_runways_df.count():>12,} rows")

# A value is "missing" if it is NULL, an empty string or the text "nan" (the three forms exist in OPDI)
def missing(c):
    return col(c).isNull() | (trim(col(c)) == "") | (lower(trim(col(c))) == "nan")

# 1) Missing values in flight_list
flight_cols = ["id", "icao24", "flt_id", "first_seen", "last_seen", "adep", "ades", "typecode", "icao_operator"]
res = bronze_flights_df.select([spark_sum(missing(c).cast("int")).alias(c) for c in flight_cols]).first().asDict()
print("\nMissing values in flight_list (NULL, '' or 'nan'):")
for c in flight_cols:
    print(f"  {c:15s} {res[c]:>10,}  ({res[c] / flights_count * 100:5.2f} %)")

# 2) Timestamps are strings in Bronze: look at their format before casting them
print("\nTimestamp examples:", bronze_flights_df.select("first_seen").first()[0], "|",
      bronze_events_df.select("event_time").first()[0])

# 3) Temporal anomalies and duplicate keys in flight_list
inversions = bronze_flights_df.filter(to_timestamp("first_seen") > to_timestamp("last_seen")).count()
dup_ids = flights_count - bronze_flights_df.select("id").distinct().count()
print(f"Flights with first_seen > last_seen: {inversions:,}")
print(f"Duplicate flight ids               : {dup_ids:,}")

# 4) Event types: exact values used later in Silver and Gold
print("\nEvent types:")
bronze_events_df.groupBy("type").count().orderBy(desc("count")).show(40, truncate=False)

# 5) Keys of the JSON column `info`, per event type (0.1 % sample)
print("Keys found in the JSON column `info` (0.1 % sample):")
(bronze_events_df.sample(fraction=0.001, seed=42)
    .select("type", explode(map_keys(from_json("info", MapType(StringType(), StringType())))).alias("info_key"))
    .groupBy("type", "info_key").count()
    .orderBy("type", "info_key")
    .show(100, truncate=False))
print("Example of `info` for an 'entry-runway' event:",
      bronze_events_df.filter(col("type") == "entry-runway").select("info").first()[0])

# 6) Altitude unit and anomalies (FL100 = 10,000 ft = 3,048 m), missing `info`
alt = bronze_events_df.agg(
    avg(when(col("type") == "first-xing-fl100", col("altitude"))).alias("fl100"),
    spark_sum((col("altitude") < 0).cast("int")).alias("negative"),
    spark_sum(missing("info").cast("int")).alias("info_missing"),
).first()
if alt["fl100"] is not None:
    print(f"\nMean altitude of 'first-xing-fl100' events: {alt['fl100']:,.0f}  (~10,000 if feet, ~3,048 if metres)")
print(f"Events with a negative altitude: {alt['negative']:,} | events without `info`: {alt['info_missing']:,}")

# 7) Referential integrity: do the flights referenced by events exist in flight_list?
event_flights = bronze_events_df.select("flight_id").distinct()
n_event_flights = event_flights.count()
matched = event_flights.join(bronze_flights_df.select(col("id").alias("flight_id")), "flight_id", "left_semi").count()
print(f"\nDistinct flights in flight_events: {n_event_flights:,} | found in flight_list: {matched:,}"
      f" ({matched / max(n_event_flights, 1) * 100:.2f} %)")

### Part 3.2: Data Model


The Silver layer turns the Bronze tables into typed, cleaned and linked tables. **Grain: one row per flight**,
with its events as a child table and two reference dimensions.

| Silver table | Grain | Key | Main columns |
|---|---|---|---|
| `silver/flights` | one flight | `flight_id` (long) | `callsign`, `transponder_icao24`, `registration`, `aircraft_typecode`, `aircraft_model`, `aircraft_class`, `operator_icao`, `origin_icao`, `dest_icao`, `route`, `flight_category`, origin/destination airport attributes, `distance_km` (great circle), `first_seen_utc`, `last_seen_utc` (timestamp), `flight_date`, `duration_min`, flags `has_adep`, `has_ades`, `is_short_flight` |
| `silver/flight_events` | one event of a flight | `event_id` (long), FK `flight_id` | `event_type`, `event_timestamp_utc`, `event_date`, `latitude`, `longitude`, `altitude_ft`, flag `is_altitude_valid`, `source`, `info` (JSON parsed as `map<string,string>`) |
| `silver/airports` | one airport with a 4-character ident (ICAO code when it exists) | `airport_icao` | name, type, country, continent, municipality, coordinates, elevation, IATA code |
| `silver/runways` | one runway of a known airport | `airport_icao` + `le_ident`/`he_ident` | length, width, surface, closed flag, headings |
| `quarantine/flights`, `quarantine/flight_events` | one rejected row | as above | all columns + `_quarantine_reason`, `_silver_flagged_at_utc` |

Every Silver row keeps its Bronze lineage (`_source_file`, `_source_period`, `_ingestion_run_id`, `_source_sha256`).

**Event types** (OPDI v0.0.2 methodology, checked in 3.1): `take-off`, `landing`, `top-of-climb`, `top-of-descent`,
`level-start`, `level-end`, `first-xing-fl50/70/100/245`, `last-xing-fl50/70/100/245`, and the airport events
`entry-runway`, `exit-runway`, `entry-taxiway`, `exit-taxiway`, `entry-parking_position`, `exit-parking_position`.

**Cleaning rules**
- **Reject (quarantine, with a reason):** missing flight id or transponder address, unparsable timestamps,
  `first_seen` > `last_seen`, flight longer than 24 h; events with no flight id, no type, unparsable time,
  coordinates out of range, or whose flight is not a valid Silver flight (`ORPHAN_EVENT`).
- **Flag and keep:** unknown departure/arrival airport (`has_adep`, `has_ades`; about half of the flights, still
  valid flights), flights shorter than 2 minutes (`is_short_flight`), implausible altitudes (`is_altitude_valid`).
- **Correct:** `""`, `"nan"` and NULL are turned into a single NULL; strings are trimmed; codes are upper-cased;
  airport codes that are not 4 characters long are set to NULL.
- **Out of scope (dropped, counted):** flights that started and events that happened outside [June 1, July 29[.
- **Duplicates:** one row per `flight_id` (the most complete one); one event per (`flight_id`, `event_type`, time).

**Not in Silver (intentionally):** the `measurements` table (cumulative distance flown and time passed since
`first_seen`, per event) is not needed for the Gold question (airport operations) and stays in Bronze; it is the
basis of a future flight-efficiency indicator. `adep_p`/`ades_p`, `dof` and `version` are dropped.

**Partitioning:** flights by `flight_year`/`flight_month`; events and quarantine tables by `_source_period`
(10-day window or month), the same layout as Bronze.

### Part 3.3: Standardization

One explicit target schema per table: snake_case names, timestamps cast from strings (UTC), upper-case codes,
a single NULL for missing values, and the `info` JSON parsed into a map. The scope filter (June 1 – July 28) is
prepared here as a column, so that out-of-scope rows can be counted in 3.8.

In [ ]:
# Missing values: '', 'nan' and NULL all become NULL (a single representation); strings are trimmed
def clean_str(c):
    v = trim(col(c))
    return when((v == "") | (lower(v) == "nan"), None).otherwise(v)

# ICAO airport code: upper case, 4 characters, otherwise NULL
def icao_code(c):
    v = upper(clean_str(c))
    return when(length(v) == 4, v)

SCOPE_START = lit(str(START)).cast("timestamp")     # 2026-06-01 00:00 UTC (inclusive)
SCOPE_END   = lit(str(END)).cast("timestamp")       # 2026-07-29 00:00 UTC (exclusive)

def in_scope(ts):
    # rows with an unparsable time are kept here, so that the quality rules can send them to quarantine
    return ts.isNull() | ((ts >= SCOPE_START) & (ts < SCOPE_END))

# 1) Airports (OurAirports): `ident` is the ICAO code when the airport has one
std_airports = (bronze_airports_df
    .filter(length(trim(col("ident"))) == 4)
    .select(
        upper(trim(col("ident"))).alias("airport_icao"),
        clean_str("name").alias("airport_name"),
        lower(clean_str("type")).alias("airport_type"),
        upper(clean_str("iso_country")).alias("iso_country"),
        upper(clean_str("continent")).alias("continent"),
        clean_str("municipality").alias("municipality"),
        col("latitude_deg").cast(DoubleType()).alias("latitude"),
        col("longitude_deg").cast(DoubleType()).alias("longitude"),
        col("elevation_ft").cast(DoubleType()).alias("elevation_ft"),
        upper(clean_str("iata_code")).alias("iata_code"))
    .dropDuplicates(["airport_icao"]))

# 2) Runways (OurAirports), only for airports kept above
std_runways = (bronze_runways_df
    .select(
        upper(trim(col("airport_ident"))).alias("airport_icao"),
        upper(clean_str("le_ident")).alias("le_ident"),
        upper(clean_str("he_ident")).alias("he_ident"),
        col("length_ft").cast(DoubleType()).alias("length_ft"),
        col("width_ft").cast(DoubleType()).alias("width_ft"),
        upper(clean_str("surface")).alias("surface"),
        (trim(col("closed")) == "1").alias("is_closed"),
        col("le_heading_degT").cast(DoubleType()).alias("le_heading_deg"),
        col("he_heading_degT").cast(DoubleType()).alias("he_heading_deg"))
    .join(std_airports.select("airport_icao"), "airport_icao", "left_semi"))

print(f"Airports kept: {std_airports.count():,} | runways kept: {std_runways.count():,}")

# 3) Flights: explicit target schema (one schema variant only, checked in 2.2)
std_flights = (bronze_flights_df
    .select(
        col("id").cast(LongType()).alias("flight_id"),
        lower(clean_str("icao24")).alias("transponder_icao24"),
        upper(clean_str("flt_id")).alias("callsign"),
        upper(clean_str("registration")).alias("registration"),
        icao_code("adep").alias("origin_icao"),
        icao_code("ades").alias("dest_icao"),
        upper(clean_str("typecode")).alias("aircraft_typecode"),
        clean_str("model").alias("aircraft_model"),
        upper(clean_str("icao_aircraft_class")).alias("aircraft_class"),
        upper(clean_str("icao_operator")).alias("operator_icao"),
        to_timestamp("first_seen").alias("first_seen_utc"),       # session time zone = UTC (1.2)
        to_timestamp("last_seen").alias("last_seen_utc"),
        "_source_file", "_source_period", "_ingestion_run_id", "_source_sha256")
    .withColumn("flight_date", to_date("first_seen_utc"))
    .withColumn("flight_year", year("first_seen_utc"))
    .withColumn("flight_month", month("first_seen_utc"))
    .withColumn("duration_min",
                spark_round((unix_timestamp("last_seen_utc") - unix_timestamp("first_seen_utc")) / 60.0, 2))
    .withColumn("_in_scope", in_scope(col("first_seen_utc"))))

# 4) Flight events: `info` JSON parsed into a map (no assumption on its keys), "nan" values removed
std_events = (bronze_events_df
    .select(
        col("id").cast(LongType()).alias("event_id"),
        col("flight_id").cast(LongType()).alias("flight_id"),
        lower(clean_str("type")).alias("event_type"),
        to_timestamp("event_time").alias("event_timestamp_utc"),
        col("latitude").cast(DoubleType()).alias("latitude"),
        col("longitude").cast(DoubleType()).alias("longitude"),
        col("altitude").cast(DoubleType()).alias("altitude_ft"),
        clean_str("source").alias("source"),
        from_json(clean_str("info"), MapType(StringType(), StringType())).alias("info"),
        "_source_file", "_source_period", "_ingestion_run_id", "_source_sha256")
    .withColumn("info", expr("map_filter(info, (k, v) -> v IS NOT NULL AND lower(v) <> 'nan')"))
    .withColumn("event_date", to_date("event_timestamp_utc"))
    .withColumn("_in_scope", in_scope(col("event_timestamp_utc"))))

print("Target schemas defined (transformations are lazy: nothing is computed yet).")

### Part 3.4: Quality and Quarantine rules

Rejected rows go to quarantine with the first rule they fail; incomplete but valid rows are kept with a flag.

In [ ]:
# ---- Flights -------------------------------------------------------------------------------------
# Reject rules: the first failing rule gives the quarantine reason (NULL = valid)
flights_flagged = (std_flights
    .filter(col("_in_scope")).drop("_in_scope")
    .withColumn("_quarantine_reason",
        when(col("flight_id").isNull(), "MISSING_FLIGHT_ID")
        .when(col("transponder_icao24").isNull(), "MISSING_ICAO24")
        .when(col("first_seen_utc").isNull() | col("last_seen_utc").isNull(), "UNPARSABLE_TIMESTAMP")
        .when(col("first_seen_utc") > col("last_seen_utc"), "INVALID_CHRONOLOGY")
        .when(col("duration_min") > 1440, "DURATION_GT_24H"))
    # Flag-and-keep rules: the flight is valid, but an attribute is incomplete or suspicious
    .withColumn("has_adep", col("origin_icao").isNotNull())
    .withColumn("has_ades", col("dest_icao").isNotNull())
    .withColumn("is_short_flight", col("duration_min") < 2)
    .withColumn("_silver_flagged_at_utc", current_timestamp())
    .persist())     # ~3.3 M rows, reused several times below: cached once instead of re-read from Bronze

flights_valid      = flights_flagged.filter(col("_quarantine_reason").isNull())
flights_quarantine = flights_flagged.filter(col("_quarantine_reason").isNotNull())

print("Flights in scope, by quality outcome:")
(flights_flagged.groupBy(coalesce(col("_quarantine_reason"), lit("VALID")).alias("outcome"))
    .count().orderBy(desc("count")).show(truncate=False))

# ---- Flight events -------------------------------------------------------------------------------
# An event is only kept if its flight is a valid Silver flight (referential integrity)
valid_flight_ids = flights_valid.select("flight_id").distinct().withColumn("_flight_known", lit(True))

events_flagged = (std_events
    .filter(col("_in_scope")).drop("_in_scope")
    .join(valid_flight_ids, "flight_id", "left")
    .withColumn("_quarantine_reason",
        when(col("flight_id").isNull(), "MISSING_FLIGHT_ID")
        .when(col("event_type").isNull(), "MISSING_EVENT_TYPE")
        .when(col("event_timestamp_utc").isNull(), "UNPARSABLE_TIMESTAMP")
        .when(~col("latitude").between(-90, 90) | ~col("longitude").between(-180, 180), "INVALID_COORDINATES")
        .when(col("_flight_known").isNull(), "ORPHAN_EVENT"))
    .drop("_flight_known")
    # Flag and keep: implausible altitude (below the lowest airports or above airliner ceilings)
    .withColumn("is_altitude_valid", col("altitude_ft").between(-1500, 60000))
    .withColumn("_silver_flagged_at_utc", current_timestamp()))

events_valid      = events_flagged.filter(col("_quarantine_reason").isNull())
events_quarantine = events_flagged.filter(col("_quarantine_reason").isNotNull())

# The ~86 M events are only processed when they are written (3.7); their quality figures are
# computed afterwards from the written tables (3.8), instead of re-scanning Bronze for every count.
print("Event rules defined: they are applied when the Silver and quarantine tables are written (3.7).")

### Part 3.5: Deduplication

In [ ]:
# Flights: one row per flight_id. If a flight appears twice (e.g. in two monthly files), keep the most
# complete row (both airports known first), then the one from the most recent file.
flight_window = Window.partitionBy("flight_id").orderBy(
    (col("has_adep").cast("int") + col("has_ades").cast("int")).desc(),
    col("_source_period").desc())

flights_deduped = (flights_valid
    .withColumn("_row_num", row_number().over(flight_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num"))

# Events: same flight, same type, same time = same event.
# dropDuplicates is a hash aggregation (no sort), cheaper than a row_number() window on ~86 M rows.
# The events are already shuffled by flight_id for the join of 3.4, and Spark can reuse that partitioning.
events_deduped = events_valid.dropDuplicates(["flight_id", "event_type", "event_timestamp_utc"])

n_before, n_after = flights_valid.count(), flights_deduped.count()     # cached: cheap
print(f"Flights: {n_before:,} -> {n_after:,} ({n_before - n_after:,} duplicates removed)")
print("Events : duplicates removed are counted in 3.8, from the written tables.")

### Part 3.6: Enrichment

Flights are joined with the OurAirports dimension (departure and arrival), which gives the route, the flight
category (domestic / international) and the great-circle distance.

In [ ]:
# Airport dimension, renamed for the departure side and for the arrival side
def airport_side(prefix):
    return std_airports.select(
        col("airport_icao").alias(f"{prefix}_icao_key"),
        col("airport_name").alias(f"{prefix}_airport_name"),
        col("iso_country").alias(f"{prefix}_country"),
        col("municipality").alias(f"{prefix}_municipality"),
        col("latitude").alias(f"{prefix}_lat"),
        col("longitude").alias(f"{prefix}_lon"))

# The airports table is small (tens of thousands of rows): broadcast join, no shuffle of the flights
flights_enriched = (flights_deduped
    .join(broadcast(airport_side("origin")), col("origin_icao") == col("origin_icao_key"), "left")
    .join(broadcast(airport_side("dest")),   col("dest_icao") == col("dest_icao_key"), "left")
    .drop("origin_icao_key", "dest_icao_key")
    .withColumn("route", concat_ws("-", coalesce(col("origin_icao"), lit("????")),
                                         coalesce(col("dest_icao"), lit("????"))))
    .withColumn("flight_category",
        when(col("origin_country").isNull() | col("dest_country").isNull(), "UNKNOWN")
        .when(col("origin_country") == col("dest_country"), "DOMESTIC")
        .otherwise("INTERNATIONAL"))
    # Great-circle distance between the two airports (haversine formula, km)
    .withColumn("_a",
        sin(radians(col("dest_lat") - col("origin_lat")) / 2) ** 2 +
        cos(radians(col("origin_lat"))) * cos(radians(col("dest_lat"))) *
        sin(radians(col("dest_lon") - col("origin_lon")) / 2) ** 2)
    .withColumn("distance_km", spark_round(lit(2 * 6371.0) * asin(sqrt(col("_a"))), 2))
    .drop("_a")
    .withColumn("_silver_processed_at_utc", current_timestamp()))

silver_flights = flights_enriched.select(
    "flight_id", "callsign", "transponder_icao24", "registration",
    "aircraft_typecode", "aircraft_model", "aircraft_class", "operator_icao",
    "origin_icao", "dest_icao", "route", "flight_category",
    "origin_airport_name", "origin_country", "origin_municipality", "origin_lat", "origin_lon",
    "dest_airport_name", "dest_country", "dest_municipality", "dest_lat", "dest_lon",
    "distance_km", "duration_min", "first_seen_utc", "last_seen_utc",
    "flight_date", "flight_year", "flight_month",
    "has_adep", "has_ades", "is_short_flight",
    "_silver_processed_at_utc", "_source_file", "_source_period", "_ingestion_run_id", "_source_sha256")

silver_events = events_deduped.select(
    "event_id", "flight_id", "event_type", "event_timestamp_utc", "event_date",
    "latitude", "longitude", "altitude_ft", "is_altitude_valid", "source", "info",
    current_timestamp().alias("_silver_processed_at_utc"),
    "_source_file", "_source_period", "_ingestion_run_id", "_source_sha256")

silver_flights.select("flight_id", "callsign", "route", "flight_category", "distance_km", "duration_min").show(5, truncate=False)

### Part 3.7: Writing certified data and quarantine data

Silver is rebuilt as a whole for the project scope (`partitionOverwriteMode=static`: the previous version of each
table is replaced, including partitions that no longer exist). The ~86 M events are processed here, in two passes
only (valid events, then rejected events), instead of once per `count()`.

In [ ]:
def write_table(df, path, partition_cols=None):
    t_start = time.time()
    writer = df.write.mode("overwrite").option("partitionOverwriteMode", "static")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(str(path))
    print(f"  {str(path):55s} {time.time() - t_start:7.1f} s")

print("Silver tables:")
write_table(silver_flights, SILVER_DIR / "flights", ["flight_year", "flight_month"])
write_table(std_airports,   SILVER_DIR / "airports")
write_table(std_runways,    SILVER_DIR / "runways")
write_table(silver_events,  SILVER_DIR / "flight_events", ["_source_period"])

print("Quarantine tables:")
write_table(flights_quarantine, QUARANTINE_DIR / "flights", ["_source_period"])
write_table(events_quarantine,  QUARANTINE_DIR / "flight_events", ["_source_period"])

flights_flagged.unpersist()

def size_gb(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*.parquet")) / 1e9

print(f"\nSilver: {size_gb(SILVER_DIR):.2f} GB | quarantine: {size_gb(QUARANTINE_DIR):.3f} GB"
      f" | free space left: {free_gb():.1f} GB")

### Part 3.8: Before-and-after evidence of quality

All figures are computed from the tables written in 3.7, so they describe exactly what was published.
Row accounting: **Bronze rows = out of scope + quarantined + duplicates removed + Silver rows.**

In [ ]:
silver_fl = spark.read.parquet(str(SILVER_DIR / "flights"))
silver_ev = spark.read.parquet(str(SILVER_DIR / "flight_events"))
# explicit schema: a quarantine table can be empty (no Parquet file to infer the schema from)
quar_fl   = spark.read.schema(flights_quarantine.schema).parquet(str(QUARANTINE_DIR / "flights"))
quar_ev   = spark.read.schema(events_quarantine.schema).parquet(str(QUARANTINE_DIR / "flight_events"))

# 1) Row accounting, Bronze -> Silver
print("--- 1. Row accounting ---")
print(f"{'':22s} {'flights':>14s} {'flight_events':>16s}")
acc = {}
for name, bronze_n, std_df, quar_df, silver_df in [
        ("flights", flights_count, std_flights, quar_fl, silver_fl),
        ("events",  events_count,  std_events,  quar_ev, silver_ev)]:
    out_n = std_df.filter(~col("_in_scope")).count()
    q_n, s_n = quar_df.count(), silver_df.count()
    acc[name] = dict(bronze=bronze_n, out=out_n, quar=q_n, dup=bronze_n - out_n - q_n - s_n, silver=s_n)
for label, key in [("Bronze rows", "bronze"), ("Out of scope", "out"), ("Quarantined", "quar"),
                   ("Duplicates removed", "dup"), ("Silver rows", "silver")]:
    f, e = acc["flights"][key], acc["events"][key]
    print(f"{label:22s} {f:>14,} {e:>16,}")
print("Accounting consistent:", all(v["dup"] >= 0 for v in acc.values()))

# 2) Quarantine reasons
print("\n--- 2. Quarantine reasons ---")
quar_fl.groupBy("_quarantine_reason").count().orderBy(desc("count")).show(truncate=False)
quar_ev.groupBy("_quarantine_reason").count().orderBy(desc("count")).show(truncate=False)

# 3) Missing values: three forms in Bronze (NULL, '', 'nan') -> one form in Silver (NULL)
print("--- 3. Missing values, before vs after ---")
pairs = [("flt_id", "callsign"), ("icao_operator", "operator_icao"), ("typecode", "aircraft_typecode"),
         ("adep", "origin_icao"), ("ades", "dest_icao")]
b = bronze_flights_df.select(
    *[spark_sum(col(src).isNull().cast("int")).alias(f"{src}_null") for src, _ in pairs],
    *[spark_sum((trim(col(src)) == "").cast("int")).alias(f"{src}_empty") for src, _ in pairs],
    *[spark_sum((lower(trim(col(src))) == "nan").cast("int")).alias(f"{src}_nan") for src, _ in pairs]).first()
s = silver_fl.select(*[spark_sum(col(dst).isNull().cast("int")).alias(dst) for _, dst in pairs]).first()
n_silver = acc["flights"]["silver"]
print(f"{'Bronze column':15s} {'NULL':>9s} {'empty':>9s} {'nan':>9s}   {'Silver column':18s} {'NULL':>9s}")
for src, dst in pairs:
    print(f"{src:15s} {b[src + '_null'] / flights_count:9.2%} {b[src + '_empty'] / flights_count:9.2%}"
          f" {b[src + '_nan'] / flights_count:9.2%}   {dst:18s} {s[dst] / n_silver:9.2%}")

# 4) Completeness and flags in Silver
print("\n--- 4. Silver flights: airports known and flags ---")
silver_fl.agg(
    spark_round(avg((col("has_adep") & col("has_ades")).cast("int")) * 100, 2).alias("pct_both_airports_known"),
    spark_round(avg(col("is_short_flight").cast("int")) * 100, 2).alias("pct_short_flights"),
    spark_round(avg(col("origin_lat").isNotNull().cast("int")) * 100, 2).alias("pct_origin_found_in_ourairports"),
).show(truncate=False)
silver_fl.groupBy("flight_category").count().orderBy(desc("count")).show(truncate=False)

# 5) Referential integrity: 3.1 showed events whose flight was missing; in Silver it must be 0
orphans = (silver_ev.select("flight_id").distinct()
           .join(silver_fl.select("flight_id"), "flight_id", "left_anti").count())
print(f"--- 5. Silver events whose flight is not in Silver flights: {orphans:,} ---")

# 6) Silver events: types and altitude flag
silver_ev.groupBy("event_type").count().orderBy(desc("count")).show(40, truncate=False)
silver_ev.agg(spark_round(avg(col("is_altitude_valid").cast("int")) * 100, 2).alias("pct_altitude_valid")).show()

# 7) Types before vs after
print("--- 7. Types, Bronze -> Silver ---")
for src, dst, df_b, df_s in [("first_seen", "first_seen_utc", bronze_flights_df, silver_fl),
                             ("event_time", "event_timestamp_utc", bronze_events_df, silver_ev),
                             ("info", "info", bronze_events_df, silver_ev)]:
    print(f"  {src:12s} {dict(df_b.dtypes)[src]:10s} -> {dst:20s} {dict(df_s.dtypes)[dst]}")

### Part 3.9: Performance

Optimisations used in the Silver layer, and measured evidence for the two main choices:
- **Broadcast joins** for the small airport dimension (no shuffle of the flights): checked in the physical plan.
- **Cache** of the flagged flights (3.3 M rows, used 5 times) instead of re-reading Bronze for each use.
- **No `count()` on the 86 M events before the write**: quality figures are computed from the written tables.
- **`dropDuplicates` instead of a `row_number()` window** for the events: hash aggregation, no global sort.
- **Partition pruning**: Silver events are partitioned by `_source_period`, so a query on one 10-day window reads one
  partition only.

The measurements below are run on one 10-day window (~15 M events) to keep them short.

In [ ]:
def timed(label, df):
    t_start = time.time()
    df.write.format("noop").mode("overwrite").save()      # runs the whole query without writing anything
    elapsed = time.time() - t_start
    print(f"  {label:50s} {elapsed:6.1f} s")
    return elapsed

def plan_lines(df, keyword):
    plan = df._jdf.queryExecution().executedPlan().toString()
    return [line.strip() for line in plan.split("\n") if keyword in line]

# 1) Broadcast join of the airports dimension (physical plan of the Silver flights)
print("--- 1. Joins in the plan of silver_flights ---")
for line in plan_lines(silver_flights, "Join"):
    print("  ", line[:120])

# 2) Joining ~15 M events with the valid flight ids: sort-merge join vs broadcast join (measured)
print("\n--- 2. Events x valid flights join, one 10-day window ---")
one_window = (spark.read.parquet(str(BRONZE_DIR / "flight_events"))
              .filter(col("_source_period") == 20260709).select("flight_id", "type"))
ids = spark.read.parquet(str(SILVER_DIR / "flights")).select("flight_id")
one_window.count()                                                   # warm-up (file cache)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)         # forbid automatic broadcast
t_smj = timed("sort-merge join (shuffle of both sides)", one_window.join(ids, "flight_id", "left_semi"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
t_bhj = timed("broadcast join (flight ids sent to every task)", one_window.join(broadcast(ids), "flight_id", "left_semi"))
print(f"  -> speed-up of the broadcast join: x{t_smj / max(t_bhj, 0.01):.1f}")

# 3) Partition pruning on Silver events: same 10-day window, filter on the partition column vs on a normal column
print("\n--- 3. Partition pruning on silver/flight_events ---")
ev = spark.read.parquet(str(SILVER_DIR / "flight_events"))
pruned     = ev.filter(col("_source_period") == 20260709)
not_pruned = ev.filter((col("event_timestamp_utc") >= "2026-07-09") & (col("event_timestamp_utc") < "2026-07-19"))
for line in plan_lines(pruned, "PartitionFilters"):
    print("  ", line[:160])
t_p  = timed("filter on the partition column (_source_period)", pruned)
t_np = timed("filter on event_timestamp_utc (all partitions scanned)", not_pruned)
print(f"  -> speed-up from partition pruning: x{t_np / max(t_p, 0.01):.1f}")

# 4) Configuration in use
print("\n--- 4. Spark configuration ---")
for key in ["spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions", "spark.driver.memory",
            "spark.sql.autoBroadcastJoinThreshold"]:
    print(f"  {key:40s} {spark.conf.get(key, 'default')}")
print(f"  free disk space: {free_gb():.1f} GB")

## Part 4: Gold Layer
### Part 4.1: Construction

**Question served:** *how busy is each European airport, how are its runways used and how long do ground phases
take, by day and by hour?* (showcase airports: Toulouse-Blagnac LFBO and Stavanger ENZV).

Gold tables are small, denormalised and aggregated, so a dashboard can query them without joins.

| Gold table | Grain | Status |
|---|---|---|
| `gold/airport_daily_ops` | airport × day | built below |
| `gold/airport_hourly_ops` | airport × day × hour | built below |
| `gold/runway_usage` | airport × runway × day | built below |
| `gold/airport_ground_times` | airport × day: taxi-out / taxi-in, with a coverage column | built below |

**Movement times.** A departure is timed by the flight's `take-off` event, and an arrival by its `landing` event.
When the event is missing, `first_seen_utc` / `last_seen_utc` are used instead, and `pct_time_from_event` shows how
often the precise time was available. All times are **UTC** (not airport local time).

**Gold pipeline structure:**
- **4.1 Construction:**
  - `airport_hourly_ops` and `airport_daily_ops`: Traffic volumes and operational metrics.
  - `runway_usage`: Usage counts by runway joined with physical characteristics from `silver/runways`.
  - `airport_ground_times`: Taxi-out (`exit-parking_position` -> `take-off`) and taxi-in (`landing` -> `entry-parking_position`) durations, percentiles and coverage.
- **4.2 Validation:** Full reconciliation with Silver, integrity range checks, and showcase checks on LFBO and ENZV.
- **4.3 Publication:** Creation of `_SUCCESS` marker and audit metadata in `gold/run_status`.


In [ ]:
# Silver inputs are read from disk, so Gold can be rebuilt without re-running Silver
silver_fl = spark.read.parquet(str(SILVER_DIR / "flights"))
silver_ev = spark.read.parquet(str(SILVER_DIR / "flight_events"))
silver_ap = spark.read.parquet(str(SILVER_DIR / "airports"))

GOLD_START = lit(str(START)).cast("timestamp")      # scope [START, END[, as in Silver
GOLD_END   = lit(str(END)).cast("timestamp")

# 1) Take-off and landing time of each flight, from its events (only 2 event types are read)
runway_times = (silver_ev
    .filter(col("event_type").isin("take-off", "landing"))
    .groupBy("flight_id")
    .agg(spark_min(when(col("event_type") == "take-off", col("event_timestamp_utc"))).alias("takeoff_utc"),
         spark_max(when(col("event_type") == "landing", col("event_timestamp_utc"))).alias("landing_utc")))

flights_timed = (silver_fl
    .select("flight_id", "origin_icao", "dest_icao", "operator_icao", "aircraft_typecode",
            "aircraft_class", "flight_category", "first_seen_utc", "last_seen_utc")
    .join(runway_times, "flight_id", "left")
    .persist())          # used twice (departures and arrivals): the events are scanned only once

# 2) One row per airport movement: a flight gives a departure at its origin and an arrival at its destination
def movements(kind, airport_col, event_col, fallback_col):
    return (flights_timed
        .filter(col(airport_col).isNotNull())
        .select("flight_id",
                col(airport_col).alias("airport_icao"),
                lit(kind).alias("movement_type"),
                coalesce(col(event_col), col(fallback_col)).alias("movement_time_utc"),
                col(event_col).isNotNull().alias("time_from_event"),
                "operator_icao", "aircraft_typecode", "aircraft_class", "flight_category"))

airport_movements = (movements("DEP", "origin_icao", "takeoff_utc", "first_seen_utc")
    .unionByName(movements("ARR", "dest_icao", "landing_utc", "last_seen_utc"))
    .filter((col("movement_time_utc") >= GOLD_START) & (col("movement_time_utc") < GOLD_END))
    .withColumn("movement_date", to_date("movement_time_utc"))
    .withColumn("movement_hour", hour("movement_time_utc"))
    # ICAO aircraft class, e.g. "L2J" = landplane, 2 engines, Jet: the 3rd character is the engine type
    .withColumn("is_jet", when(col("aircraft_class").isNotNull(), col("aircraft_class").substr(3, 1) == "J"))
    .persist())          # reused by all the Gold tables (and by the runway / ground-time tables to come)

print("Airport movements:")
(airport_movements.groupBy("movement_type")
    .agg(count("*").alias("movements"),
         spark_round(avg(col("time_from_event").cast("int")) * 100, 1).alias("pct_time_from_event"))
    .show())
flights_timed.unpersist()

In [ ]:
airports_dim = silver_ap.select("airport_icao", "airport_name", "iso_country", "municipality")

def dep_arr_counts():
    return [spark_sum((col("movement_type") == "DEP").cast("int")).alias("departures"),
            spark_sum((col("movement_type") == "ARR").cast("int")).alias("arrivals"),
            count("*").alias("movements")]

# 1) airport x day x hour
airport_hourly_ops = (airport_movements
    .groupBy("airport_icao", "movement_date", "movement_hour")
    .agg(*dep_arr_counts())
    .join(broadcast(airports_dim), "airport_icao", "left")
    .withColumn("_gold_built_at_utc", current_timestamp()))

# 2) airport x day (averages ignore NULL: unknown aircraft class / unknown country are left out of the shares)
peak_hour = (airport_hourly_ops.groupBy("airport_icao", "movement_date")
             .agg(spark_max("movements").alias("peak_hour_movements")))

airport_daily_ops = (airport_movements
    .groupBy("airport_icao", "movement_date")
    .agg(*dep_arr_counts(),
         countDistinct("operator_icao").alias("distinct_operators"),
         countDistinct("aircraft_typecode").alias("distinct_aircraft_types"),
         spark_round(avg(col("is_jet").cast("int")) * 100, 1).alias("pct_jet"),
         spark_round(avg(when(col("flight_category") != "UNKNOWN",
                              (col("flight_category") == "INTERNATIONAL").cast("int"))) * 100, 1).alias("pct_international"),
         spark_round(avg(col("time_from_event").cast("int")) * 100, 1).alias("pct_time_from_event"))
    .join(peak_hour, ["airport_icao", "movement_date"], "left")
    .join(broadcast(airports_dim), "airport_icao", "left")
    .withColumn("day_of_week", dayofweek("movement_date"))      # 1 = Sunday ... 7 = Saturday
    .withColumn("_gold_built_at_utc", current_timestamp()))

# 3) Write. Gold tables are small: few output files (coalesce) instead of one per shuffle partition
GOLD_DIR.mkdir(parents=True, exist_ok=True)
for name, df, n_files in [("airport_hourly_ops", airport_hourly_ops, 4), ("airport_daily_ops", airport_daily_ops, 1)]:
    t_start = time.time()
    (df.coalesce(n_files).write.mode("overwrite").option("partitionOverwriteMode", "static")
       .parquet(str(GOLD_DIR / name)))
    n_rows = spark.read.parquet(str(GOLD_DIR / name)).count()
    print(f"gold/{name:20s} {n_rows:>10,} rows  ({time.time() - t_start:.1f} s)")

# 4) First look at the result
gold_daily = spark.read.parquet(str(GOLD_DIR / "airport_daily_ops"))
print("\nTop 10 airports by movements over the period:")
(gold_daily.groupBy("airport_icao", "airport_name", "iso_country")
    .agg(spark_sum("movements").alias("movements"), spark_round(avg("movements"), 0).alias("avg_per_day"))
    .orderBy(desc("movements")).show(10, truncate=False))

for apt in ["LFBO", "ENZV"]:
    print(f"Showcase airport {apt}: first days of the period")
    (gold_daily.filter(col("airport_icao") == apt)
        .select("movement_date", "departures", "arrivals", "movements", "peak_hour_movements",
                "distinct_operators", "pct_jet", "pct_international", "pct_time_from_event")
        .orderBy("movement_date").show(7))

In [ ]:
# --------------------------------------------------------------------------------------------------
# Part 4.1 (continued) - Mathis
# Table 1: gold/runway_usage (airport x runway x day)
# --------------------------------------------------------------------------------------------------
silver_rw = spark.read.parquet(str(SILVER_DIR / "runways"))

# 1. Filter runway events (entry-runway for takeoffs, exit-runway for landings)
# info map contains the runway identifier, typically under 'ref', 'runway', or 'name'
rw_events = (silver_ev
    .filter(col("event_type").isin("entry-runway", "exit-runway"))
    .filter(col("info").isNotNull())
    .withColumn("runway_ident", upper(trim(coalesce(col("info")["ref"], col("info")["runway"], col("info")["name"]))))
    .filter(col("runway_ident").isNotNull() & (col("runway_ident") != ""))
    .select("flight_id", "event_type", "event_timestamp_utc", "runway_ident"))

# 2. Join with airport_movements to associate with airport and movement type
rw_movements = (airport_movements
    .select("flight_id", "airport_icao", "movement_type", "movement_date")
    .join(rw_events, "flight_id", "inner")
    .filter(
        ((col("movement_type") == "DEP") & (col("event_type") == "entry-runway")) |
        ((col("movement_type") == "ARR") & (col("event_type") == "exit-runway"))
    ))

# 3. Aggregate by airport_icao x runway_ident x movement_date
runway_daily_counts = (rw_movements
    .groupBy("airport_icao", "runway_ident", "movement_date")
    .agg(
        spark_sum(when(col("movement_type") == "DEP", 1).otherwise(0)).alias("runway_departures"),
        spark_sum(when(col("movement_type") == "ARR", 1).otherwise(0)).alias("runway_arrivals"),
        count("*").alias("runway_movements")
    ))

# 4. Enrich with runway physical characteristics from silver/runways
rw_dim = (silver_rw
    .select(
        col("airport_icao").alias("rw_airport_icao"),
        col("le_ident"),
        col("he_ident"),
        col("length_ft"),
        col("width_ft"),
        col("surface"),
        col("is_closed")
    ))

runway_usage = (runway_daily_counts
    .join(
        broadcast(rw_dim),
        (col("airport_icao") == col("rw_airport_icao")) & (
            (col("runway_ident") == col("le_ident")) |
            (col("runway_ident") == col("he_ident")) |
            (col("runway_ident") == concat_ws("/", col("le_ident"), col("he_ident")))
        ),
        "left"
    )
    .drop("rw_airport_icao", "le_ident", "he_ident")
    .withColumn("_gold_built_at_utc", current_timestamp()))

# 5. Write gold/runway_usage
t_start = time.time()
(runway_usage.coalesce(2).write.mode("overwrite").option("partitionOverwriteMode", "static")
   .parquet(str(GOLD_DIR / "runway_usage")))
n_rw = spark.read.parquet(str(GOLD_DIR / "runway_usage")).count()
print(f"gold/runway_usage         {n_rw:>10,} rows  ({time.time() - t_start:.1f} s)")

# --------------------------------------------------------------------------------------------------
# Table 2: gold/airport_ground_times (airport x day: taxi-out / taxi-in durations and coverage)
# --------------------------------------------------------------------------------------------------
ground_events = (silver_ev
    .filter(col("event_type").isin("exit-parking_position", "take-off", "landing", "entry-parking_position"))
    .groupBy("flight_id")
    .agg(
        spark_min(when(col("event_type") == "exit-parking_position", col("event_timestamp_utc"))).alias("exit_parking_utc"),
        spark_min(when(col("event_type") == "take-off", col("event_timestamp_utc"))).alias("takeoff_utc"),
        spark_max(when(col("event_type") == "landing", col("event_timestamp_utc"))).alias("landing_utc"),
        spark_max(when(col("event_type") == "entry-parking_position", col("event_timestamp_utc"))).alias("entry_parking_utc")
    )
    .withColumn("taxi_out_min",
        spark_round((unix_timestamp("takeoff_utc") - unix_timestamp("exit_parking_utc")) / 60.0, 1))
    .withColumn("taxi_in_min",
        spark_round((unix_timestamp("entry_parking_utc") - unix_timestamp("landing_utc")) / 60.0, 1))
    .withColumn("valid_taxi_out", col("taxi_out_min").between(1, 120))
    .withColumn("valid_taxi_in", col("taxi_in_min").between(1, 120)))

dep_taxi = (airport_movements
    .filter(col("movement_type") == "DEP")
    .select("flight_id", "airport_icao", "movement_date")
    .join(ground_events.filter(col("valid_taxi_out")).select("flight_id", "taxi_out_min"), "flight_id", "left"))

arr_taxi = (airport_movements
    .filter(col("movement_type") == "ARR")
    .select("flight_id", "airport_icao", "movement_date")
    .join(ground_events.filter(col("valid_taxi_in")).select("flight_id", "taxi_in_min"), "flight_id", "left"))

taxi_out_daily = (dep_taxi
    .groupBy("airport_icao", "movement_date")
    .agg(
        count("*").alias("total_departures"),
        count("taxi_out_min").alias("taxi_out_measured_flights"),
        spark_round(avg("taxi_out_min"), 1).alias("avg_taxi_out_min"),
        spark_round(expr("percentile_approx(taxi_out_min, 0.5)"), 1).alias("median_taxi_out_min"),
        spark_round(expr("percentile_approx(taxi_out_min, 0.9)"), 1).alias("p90_taxi_out_min")
    ))

taxi_in_daily = (arr_taxi
    .groupBy("airport_icao", "movement_date")
    .agg(
        count("*").alias("total_arrivals"),
        count("taxi_in_min").alias("taxi_in_measured_flights"),
        spark_round(avg("taxi_in_min"), 1).alias("avg_taxi_in_min"),
        spark_round(expr("percentile_approx(taxi_in_min, 0.5)"), 1).alias("median_taxi_in_min"),
        spark_round(expr("percentile_approx(taxi_in_min, 0.9)"), 1).alias("p90_taxi_in_min")
    ))

airport_ground_times = (taxi_out_daily
    .join(taxi_in_daily, ["airport_icao", "movement_date"], "outer")
    .join(broadcast(airports_dim), "airport_icao", "left")
    .withColumn("pct_taxi_out_coverage",
        when(col("total_departures") > 0,
             spark_round(col("taxi_out_measured_flights") / col("total_departures") * 100, 1)).otherwise(lit(0.0)))
    .withColumn("pct_taxi_in_coverage",
        when(col("total_arrivals") > 0,
             spark_round(col("taxi_in_measured_flights") / col("total_arrivals") * 100, 1)).otherwise(lit(0.0)))
    .withColumn("_gold_built_at_utc", current_timestamp()))

t_start = time.time()
(airport_ground_times.coalesce(1).write.mode("overwrite").option("partitionOverwriteMode", "static")
   .parquet(str(GOLD_DIR / "airport_ground_times")))
n_gt = spark.read.parquet(str(GOLD_DIR / "airport_ground_times")).count()
print(f"gold/airport_ground_times {n_gt:>10,} rows  ({time.time() - t_start:.1f} s)")

# Showcase display
for apt in ["LFBO", "ENZV"]:
    print(f"\nShowcase airport {apt}: ground times (taxi-out and taxi-in)")
    (spark.read.parquet(str(GOLD_DIR / "airport_ground_times"))
        .filter(col("airport_icao") == apt)
        .select("movement_date", "avg_taxi_out_min", "median_taxi_out_min", "p90_taxi_out_min", "pct_taxi_out_coverage",
                "avg_taxi_in_min", "median_taxi_in_min", "p90_taxi_in_min", "pct_taxi_in_coverage")
        .orderBy("movement_date").show(7))


### Part 4.2: Validation

In [ ]:
# --------------------------------------------------------------------------------------------------
# Part 4.2: Validation
# --------------------------------------------------------------------------------------------------
print("=" * 70)
print("PART 4.2: GOLD LAYER VALIDATION")
print("=" * 70)

# Load generated Gold tables
gold_daily = spark.read.parquet(str(GOLD_DIR / "airport_daily_ops"))
gold_hourly = spark.read.parquet(str(GOLD_DIR / "airport_hourly_ops"))
gold_runway = spark.read.parquet(str(GOLD_DIR / "runway_usage"))
gold_ground = spark.read.parquet(str(GOLD_DIR / "airport_ground_times"))

# 1) Reconciliation with Silver / airport_movements
total_movements_input = airport_movements.count()
total_movements_daily = gold_daily.agg(spark_sum("movements")).first()[0]
total_movements_hourly = gold_hourly.agg(spark_sum("movements")).first()[0]

recon_daily_ok = (total_movements_input == total_movements_daily)
recon_hourly_ok = (total_movements_input == total_movements_hourly)

print(f"1. Movements Reconciliation:")
print(f"   Input movements (airport_movements) : {total_movements_input:>12,}")
print(f"   Daily aggregated movements           : {total_movements_daily:>12,}  -> {'PASSED' if recon_daily_ok else 'FAILED'}")
print(f"   Hourly aggregated movements          : {total_movements_hourly:>12,}  -> {'PASSED' if recon_hourly_ok else 'FAILED'}")

# 2) Range and Invariance Checks
invalid_daily = gold_daily.filter(
    (col("departures") < 0) | (col("arrivals") < 0) | (col("movements") < 0) |
    (col("movements") != (col("departures") + col("arrivals"))) |
    (col("pct_jet") < 0) | (col("pct_jet") > 100) |
    (col("pct_international") < 0) | (col("pct_international") > 100) |
    (col("pct_time_from_event") < 0) | (col("pct_time_from_event") > 100)
).count()

invalid_hourly = gold_hourly.filter(
    (col("departures") < 0) | (col("arrivals") < 0) | (col("movements") < 0) |
    (col("movements") != (col("departures") + col("arrivals")))
).count()

invalid_ground = gold_ground.filter(
    (col("pct_taxi_out_coverage") < 0) | (col("pct_taxi_out_coverage") > 100) |
    (col("pct_taxi_in_coverage") < 0) | (col("pct_taxi_in_coverage") > 100) |
    (col("avg_taxi_out_min") < 0) | (col("avg_taxi_in_min") < 0)
).count()

range_checks_ok = (invalid_daily == 0 and invalid_hourly == 0 and invalid_ground == 0)
print(f"\n2. Range & Consistency Checks:")
print(f"   airport_daily_ops invalid rows   : {invalid_daily}  -> {'PASSED' if invalid_daily == 0 else 'FAILED'}")
print(f"   airport_hourly_ops invalid rows  : {invalid_hourly}  -> {'PASSED' if invalid_hourly == 0 else 'FAILED'}")
print(f"   airport_ground_times invalid rows: {invalid_ground}  -> {'PASSED' if invalid_ground == 0 else 'FAILED'}")

# 3) Showcase Airport Verification (LFBO - Toulouse, ENZV - Stavanger)
print(f"\n3. Showcase Airport Verification (LFBO & ENZV):")
for apt in ["LFBO", "ENZV"]:
    stats = gold_daily.filter(col("airport_icao") == apt).agg(
        countDistinct("movement_date").alias("days"),
        spark_sum("movements").alias("total_movements"),
        spark_round(avg("movements"), 1).alias("avg_daily_movements"),
        spark_round(avg("pct_jet"), 1).alias("avg_pct_jet")
    ).first()
    print(f"   [{apt}] Days: {stats['days']} | Total Movements: {stats['total_movements']:,} | Avg Daily: {stats['avg_daily_movements']} mov/day | Avg Jet: {stats['avg_pct_jet']}%")

validation_passed = recon_daily_ok and recon_hourly_ok and range_checks_ok
print(f"\nOVERALL GOLD VALIDATION: {'PASSED' if validation_passed else 'FAILED'}")


### Part 4.3: Publication

In [ ]:
# --------------------------------------------------------------------------------------------------
# Part 4.3: Publication
# --------------------------------------------------------------------------------------------------
print("=" * 70)
print("PART 4.3: PUBLICATION & AUDIT TRAIL")
print("=" * 70)

if not validation_passed:
    raise RuntimeError("Publication aborted: Gold layer validation failed!")

# 1. Publish _SUCCESS marker file
success_marker_path = GOLD_DIR / "_SUCCESS"
with open(success_marker_path, "w") as f:
    f.write(f"Gold Layer Built Successfully on {utc_now().isoformat()}\n")
print(f"Created success marker: {success_marker_path}")

# 2. Build and save run_status metadata record
run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")
gold_tables_info = [
    ("airport_hourly_ops", gold_hourly.count(), sum(p.stat().st_size for p in (GOLD_DIR / "airport_hourly_ops").glob("*.parquet")) / 1e6),
    ("airport_daily_ops",  gold_daily.count(),  sum(p.stat().st_size for p in (GOLD_DIR / "airport_daily_ops").glob("*.parquet")) / 1e6),
    ("runway_usage",       gold_runway.count(), sum(p.stat().st_size for p in (GOLD_DIR / "runway_usage").glob("*.parquet")) / 1e6),
    ("airport_ground_times", gold_ground.count(), sum(p.stat().st_size for p in (GOLD_DIR / "airport_ground_times").glob("*.parquet")) / 1e6),
]

run_status_rows = []
for tbl, n_rows, size_mb in gold_tables_info:
    run_status_rows.append(Row(
        run_id=run_id,
        table_name=tbl,
        scope_start=str(START),
        scope_end=str(END),
        row_count=int(n_rows),
        size_mb=round(float(size_mb), 2),
        validation_status="PASSED",
        published_at_utc=utc_now().isoformat()
    ))

run_status_df = spark.createDataFrame(run_status_rows)
run_status_df.write.mode("append").parquet(str(GOLD_DIR / "run_status"))

print(f"\nPublished Gold Run Status (Run ID: {run_id}):")
run_status_df.select("table_name", "row_count", "size_mb", "validation_status").show(truncate=False)

total_gold_mb = sum(p.stat().st_size for p in GOLD_DIR.rglob("*.parquet")) / 1e6
print(f"Total Gold Layer Size: {total_gold_mb:.2f} MB | Free Disk Space: {free_gb():.1f} GB")
print("\n[SUCCESS] Part 4 (Gold Layer) complete and published.")


## Part 5: validation summary

## Part 6: Conclusion and limits